In [3]:
import pandas as pd
import numpy as np
from geopy.distance import distance
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
data = pd.read_csv('uber.csv')

# Drop unnecessary columns
data = data.drop(columns=['Unnamed: 0', 'key'])

# Check for null values and drop rows with any null values
data = data.dropna()

# Drop duplicate rows, if any
data = data.drop_duplicates()

# Filter out rows with invalid latitude and longitude values
data = data[(data.pickup_latitude < 90) & (data.dropoff_latitude < 90) &
            (data.pickup_latitude > -90) & (data.dropoff_latitude > -90) &
            (data.pickup_longitude < 180) & (data.dropoff_longitude < 180) &
            (data.pickup_longitude > -180) & (data.dropoff_longitude > -180)]

# Convert 'pickup_datetime' to datetime format and extract time-based features
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'], errors='coerce')
data['pickup_hour'] = data['pickup_datetime'].dt.hour
data['pickup_day'] = data['pickup_datetime'].dt.day
data['pickup_month'] = data['pickup_datetime'].dt.month
data['pickup_dayofweek'] = data['pickup_datetime'].dt.dayofweek

# Calculate distance using geopy
data['distance_miles'] = [
    round(distance((data.pickup_latitude[i], data.pickup_longitude[i]),
                   (data.dropoff_latitude[i], data.dropoff_longitude[i])).miles, 2)
    for i in data.index
]

# Prepare the data for regression
# Drop columns we no longer need
data = data.drop(columns=['pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude'])

# Filter based on suggested thresholds
data = data[(data['fare_amount'] > 0) & (data['fare_amount'] <= 113)]
data = data[(data['distance_miles'] > 0) & (data['distance_miles'] <= 45)]

# Define features and target variable
X = data[['distance_miles', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'passenger_count']]
y = data['fare_amount']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply polynomial transformation
degree = 2  # You can change the degree to experiment with different polynomial degrees
poly = PolynomialFeatures(degree=degree)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Initialize the model
poly_model = LinearRegression()

# Train the model on the polynomial-transformed training data
poly_model.fit(X_train_poly, y_train)

# Make predictions on both training and test data
y_train_pred_poly = poly_model.predict(X_train_poly)
y_test_pred_poly = poly_model.predict(X_test_poly)

# Evaluate the model on training data
train_mae_poly = mean_absolute_error(y_train, y_train_pred_poly)
train_rmse_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_poly))
train_r2_poly = r2_score(y_train, y_train_pred_poly)

# Evaluate the model on test data
test_mae_poly = mean_absolute_error(y_test, y_test_pred_poly)
test_rmse_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))
test_r2_poly = r2_score(y_test, y_test_pred_poly)

# Print results
print("Polynomial Regression (Degree 2) Training Results:")
print("Train Mean Absolute Error (MAE):", train_mae_poly)
print("Train Root Mean Squared Error (RMSE):", train_rmse_poly)
print("Train R-squared (R²):", train_r2_poly)

print("\nPolynomial Regression (Degree 2) Testing Results:")
print("Test Mean Absolute Error (MAE):", test_mae_poly)
print("Test Root Mean Squared Error (RMSE):", test_rmse_poly)
print("Test R-squared (R²):", test_r2_poly)


# # Train the model on the polynomial-transformed training data
# poly_model.fit(X_train_poly, y_train)

# # Make predictions on the polynomial-transformed test data
# y_pred_poly = poly_model.predict(X_test_poly)

# # Evaluate the model
# mae_poly = mean_absolute_error(y_test, y_pred_poly)
# rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
# r2_poly = r2_score(y_test, y_pred_poly)

# print("Polynomial Regression (Degree 2) Results:")
# print("Mean Absolute Error (MAE):", mae_poly)
# print("Root Mean Squared Error (RMSE):", rmse_poly)
# print("R-squared (R²):", r2_poly)



Polynomial Regression (Degree 2) Training Results:
Train Mean Absolute Error (MAE): 2.24987811655499
Train Root Mean Squared Error (RMSE): 4.140548594409012
Train R-squared (R²): 0.8036557178352817

Polynomial Regression (Degree 2) Testing Results:
Test Mean Absolute Error (MAE): 2.2330634983188054
Test Root Mean Squared Error (RMSE): 4.142392456682093
Test R-squared (R²): 0.8049741148418178
